# 第 02 課 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是一個用於構建 AI 代理的統一框架。它提供了一個乾淨、可組合的架構，包含四個核心構建模塊：

- **Client** – 連接至 AI 模型端點並處理通訊
- **Agent** – 將指令和工具定義包裝在客戶端中
- **Tools** – 透過模型可調用的自定義函數擴展代理功能
- **Session** – 維持多輪互動的對話歷史

在本課程中，我們將構建一個 <strong>旅遊訂票代理</strong>，使用這些概念來檢查目的地的可用性。


## 設定


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## 理解代理框架架構

Microsoft Agent 框架遵循分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. <strong>用戶端</strong> – `FoundryChatClient` 連接到 Azure OpenAI 部署。它負責身份驗證、請求格式化及回應解析。
2. <strong>代理</strong> – 透過 `provider.create_agent()` 從用戶端建立，代理結合模型訪問、指令（系統提示）與工具。
3. <strong>工具</strong> – 使用 `@tool` 裝飾的 Python 函數，代理可調用來執行操作或檢索資料。
4. <strong>會話</strong> – `AgentSession` 物件（透過 `agent.create_session()` 建立）保存對話歷史，支援多輪對話，使代理記住之前的上下文。

讓我們一步步建立每個層級。


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 使用 @tool 裝飾器新增工具

工具讓代理可以執行生成文字以外的動作。`@tool` 裝飾器會將一般的 Python 函式轉換成代理可以呼叫的功能。

重要重點：
- 使用 `Annotated[type, "description"]` 讓模型了解每個參數。
- 文件字串會成為模型看到的工具描述。
- `approval_mode="never_require"` 表示該工具會自動執行，無需使用者確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 使用工具建立代理

現在我們將客戶端、指示和工具結合成一個代理。`instructions` 作為系統提示 —— 它們定義了代理的身份和行為。


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 多輪對話與會話

一個 `AgentSession`（透過 `agent.create_session()` 建立）會追蹤對話中的所有訊息。透過在每次呼叫 `agent.run()` 時傳入相同的會話，代理人可以取得完整的對話歷史，並且能參考早期的訊息。

我們傳入 `tools=[check_destination_availability]`，讓代理人可以在每輪對話期間呼叫我們的可用性檢查工具。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 摘要

在本課程中，您探索了微軟代理框架的四大支柱：

| 概念 | 您學到了什麼 |
|---------|------------------|
| <strong>客戶端</strong> | `FoundryChatClient` 使用基於憑證的身份驗證連接到 Azure OpenAI |
| <strong>代理</strong> | `provider.create_agent()` 將模型連接與指令和名稱綁定 |
| <strong>工具</strong> | `@tool` 裝飾器將 Python 函數公開給代理調用 |
| <strong>會話</strong> | `agent.create_session()` 在多輪對話中維護對話歷史 |

這些構建模塊組合起來，創建可以進行自然對話、調用外部函數並保持上下文的代理 —— 這是後續課程中更高級代理模式的基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
